In [1]:
import pandas as pd
from openpyxl import load_workbook
import xlwings as xw
import openpyxl
import datetime
import os
import glob
import holidays  # jeśli chcesz uwzględnić święta
from pathlib import Path

In [2]:
zek103_dtypes = {
    'Werk': 'string',
    'Mat': 'string',
}

mb52_dtypes = {
    'Material': 'string',
    'Lagerort': 'string',
    'Werk': 'string',
}

z_mat_consumption_dtypes = {
    'Material': 'string',
    'movement': 'string',
}

mb51_consumption_dtypes = {
    'Materiał': 'string',
}

MB51_new_col_names = {
    'Materiał': 'Material',
    'Data księgowania': 'date',
    'Ilość': 'quantity',
    'Podst. jedn. miary': 'unit'
}

supplier_files_dict = {
    "SHC": ["CONS_246", "CONS_246_2"],
    "HESSE": ["CONS_246", "CONS_246_2"],
    "ROTO_ELZETT": ["CONS_246", "CONS_246_2"],
    "TOKOZ": ["CONS_246", "CONS_246_2"],
    "HOHAGE": ["CONS_246", "CONS_246_2"],
    "KROSNO": ["CONS_246", "CONS_246_2"],
    "FRANZEN": ["CONS_246", "CONS_246_2"],
    "ROZTOCZE": ["CONS_246", "CONS_246_2"],
    "BELATRONIC": ["CONS_246", "CONS_246_2"],
    "DEVENTER": ["CONS_246", "CONS_246_2"],
    "NMC": ["CONS_246", "CONS_246_2"],
    "NEHER": ["CONS_223", "CONS_223_2"],
    "EJOT": ["CONS_224", "CONS_224_2"],
    "SPAX": ["CONS_224", "CONS_224_2"],
    "PROFINE": ["CONS_224", "CONS_224_2"],
    "STOROPACK": ["CONS_234", "CONS_234_2"],
    "YUYAO_JILO": ["CONS_234", "CONS_234_2"],
    "BRETTHAUER": ["CONS_234", "CONS_234_2"],
    "LEO_FRANCOIS": ["CONS_234", "CONS_234_2"],
    "GALLARDO": ["CONS_234", "CONS_234_2"],
    "REISSER": ["CONS_234", "CONS_234_2"],
    "ABC_COLORE": ["CONS_228", "CONS_228_2"],
    "BACCARAT": ["CONS_228", "CONS_228_2"],
    "DOMICET": ["CONS_239", "CONS_239_2"],
    "BIZEA": ["CONS_239", "CONS_239_2"],
    "DAFA": ["CONS_239", "CONS_239_2"],
    "RETECH": ["CONS_239", "CONS_239_2"],
    "KABEL_MAIER": ["CONS_239", "CONS_239_2"],
    "GRUNEFELD": ["CONS_239", "CONS_239_2"]
}


zek103_file_path = r"\\rfmesrv5\connect\DST_SAP_Transfer\P11\PPS_LUB\05_PURCHASING_AUTOMATION\ZEK103_PUR_LUB_002.xlsx"
mb52_file_path = r"\\rfmesrv5\connect\DST_SAP_Transfer\P11\PPS_LUB\05_PURCHASING_AUTOMATION\MB52_PUR_LUB_003.xlsx"
z_mat_consumption_filepath = r"\\rfmesrv5\connect\DST_SAP_Transfer\P11\PPS_LUB\05_PURCHASING_AUTOMATION\Z_MAT_CONSUMPTION.xlsx"
mb51_consumption_filepath = r"\\rfmesrv5\connect\DST_SAP_Transfer\P11\PPS_LUB\05_PURCHASING_AUTOMATION\MB51_usage.XLSX"

main_excel_path = r'P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\PurchAutomationTemplate.xlsm'
supplier_files_directory_path_test = r'P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files'
supplier_files_directory_path = r'P:\Zakupy\O\SupplierAutomation\supplier_files'
export_files_directory_path = r"\\rfmesrv5\connect\DST_SAP_Transfer\P11\PPS_LUB\05_PURCHASING_AUTOMATION"

## Complete script

### Functions

In [9]:
def get_supplier_sap_numbers(filepath, sheet_name=None):
    """
    Returns a list of SAP numbers from an Excel file, which are found in column A
    between the row containing 'SAP' and the row containing 'Consumption'.

    :param filepath: Path to the Excel file (.xlsx)
    :param sheet_name: Name of the worksheet (if None, the active sheet is used)
    :return: List of SAP numbers (values from column A)
    """
    # Open the Excel file
    wb = openpyxl.load_workbook(filepath, keep_vba=True)
    if sheet_name:
        sheet = wb[sheet_name]
    else:
        sheet = wb.active

    # Search for the headers in column A
    row_sap = None
    row_consumption = None
    for row in range(1, sheet.max_row + 1):
        cell_value = sheet.cell(row=row, column=1).value
        if cell_value is not None:
            value_str = str(cell_value).strip().upper()
            if value_str == 'SAP' and row_sap is None:
                row_sap = row
            elif value_str == 'CONSUMPTION' and row_consumption is None:
                row_consumption = row
        if row_sap is not None and row_consumption is not None:
            break

    if row_sap is None or row_consumption is None:
        return []  # Or raise an Exception if you prefer

    # Get numbers between the found headers (exclusive)
    sap_numbers = []
    for row in range(row_sap + 1, row_consumption):
        value = sheet.cell(row=row, column=1).value
        if value is not None:
            sap_numbers.append(str(value))
    return sap_numbers

def get_zek103_data(zek103_data, plant, sap_numbers):
    zek103_data = zek103_data[zek103_data['Werk'].isin(plant)]
    zek103_data = zek103_data[zek103_data['Mat'].isin(sap_numbers)]

    zek103_data_grouped = zek103_data.groupby(['Lieferdatum', 'Mat'], as_index=False)['Best-Mg'].sum()

    zek103_data_grouped['Lieferdatum'] = pd.to_datetime(zek103_data_grouped['Lieferdatum'])  # opcjonalna konwersja na datę
    zek103_data_grouped['delayed'] = zek103_data_grouped['Lieferdatum'] < pd.Timestamp('today').normalize()

    return zek103_data_grouped

def update_excel_with_quantities(filepath, df, header_upper_bound, header_lower_bound, sheet_name, is_order_data=False):
    """
    Updates an Excel file with quantities from a DataFrame based on matching SAP numbers
    and Lieferdatum (dates).

    :param filepath: Path to the Excel file
    :param df: DataFrame with columns ['Lieferdatum', 'Mat', 'Best-Mg', 'delayed']
    """

    # Load the workbook
    wb = openpyxl.load_workbook(filepath, keep_vba=True)
    # sheet = wb.active  # Operate on the active sheet
    sheet = wb[sheet_name]

    # Find the row range for SAP numbers (between "SAP" and "Consumption")
    sap_start_row = None
    sap_end_row = None
    for row in range(1, sheet.max_row + 1):
        cell_value = sheet.cell(row=row, column=1).value  # Column A
        if cell_value is not None:
            value_str = str(cell_value).strip().upper()
            if value_str == header_upper_bound and sap_start_row is None:
                sap_start_row = row + 1  # Start after "SAP"
            elif value_str == header_lower_bound and sap_end_row is None:
                sap_end_row = row - 1  # End before "Consumption"

    if sap_start_row is None or sap_end_row is None:
        raise ValueError(f"Unable to find {header_upper_bound} and {header_lower_bound} headers in column A.")

    # Read SAP numbers from the specified range in column A
    sap_numbers = {}
    for row in range(sap_start_row, sap_end_row + 1):
        mat_number = sheet.cell(row=row, column=1).value
        if mat_number is not None:
            sap_numbers[str(mat_number)] = row

    # Read dates from the first row (headers starting from column M)
    date_columns = {'delayed': 12}  # Column with delayed orders

    for col in range(13, sheet.max_column + 1):  # Column index starts at M (13th column)
        date_value = sheet.cell(row=1, column=col).value
        if isinstance(date_value, pd.Timestamp) or isinstance(date_value, datetime.date):
            date_value = pd.Timestamp(date_value)  # Ensure it is a pandas Timestamp
        if date_value:
            date_columns[date_value.date()] = col

    # Clear data in rows between 'SAP' and 'Consumption', from column L to column L+300 (12th to 312th column)
    for row in range(sap_start_row, sap_end_row + 1):
        for col in range(12, 312):  # Column L (12th column) to 312th column
            sheet.cell(row=row, column=col).value = None

    # Iterate over the DataFrame rows
    for _, row in df.iterrows():
        lieferdatum = row['Lieferdatum']
        mat_number = row['Mat']
        quantity = row['Best-Mg']
        delayed = row['delayed']

        # Ensure lieferdatum is a date (convert if necessary)
        if isinstance(lieferdatum, str):
            lieferdatum = pd.to_datetime(lieferdatum).date()
        elif isinstance(lieferdatum, pd.Timestamp):
            lieferdatum = lieferdatum.date()

        # Match SAP number and date to find the correct cell
        if mat_number in sap_numbers and lieferdatum in date_columns:
            sap_row = sap_numbers[mat_number]
            date_col = date_columns[lieferdatum]
            if delayed and is_order_data:
                date_col = date_columns['delayed']

            # Write quantity to the matched cell
            sheet.cell(row=sap_row, column=date_col).value = quantity

    # Save the updated workbook
    wb.save(filepath)
    print(f"Excel file {os.path.basename(filepath)} updated successfully.")

def list_excel_files(directory):
    """
    Returns a list of full paths to all Excel files (.xlsx and .xlsm) in a specified directory.

    :param directory: Path to the directory to search for Excel files
    :return: List of full paths to Excel files
    """

    # Use glob to find .xlsx and .xlsm files in the directory
    xlsx_files = glob.glob(os.path.join(directory, '*.xlsx'))
    xlsm_files = glob.glob(os.path.join(directory, '*.xlsm'))

    # Combine the two lists
    excel_files = xlsx_files + xlsm_files

    return excel_files

def get_mb52_data(file_path, dtypes):

    df = pd.read_excel(file_path, dtype=dtypes)
    return df

def filter_mb52_data(df, sap_numbers, plant='2101', storage_locs='0007'):
    # Ensure storage locks is a tuple (convert if necessary)
    if isinstance(storage_locs, str):
        storage_locs = storage_locs,

    df = df[(df['Lagerort'].isin(storage_locs)) & (df['Werk'] == plant) & (df['Material'].isin(sap_numbers))]

    df_grouped = df.groupby(['Material'], as_index=False)['Frei verwendbar'].sum()

    return df_grouped

def update_excel_with_dataframe(file_path, dataframe, sap_column, frei_column, header_start, header_end, sheet_name, col_name='Frei verwendbar'):
    """
    Uzupełnia plik Excel danymi z DataFrame w określonym zakresie wierszy między nagłówkami.

    :param file_path: Ścieżka do pliku Excel
    :param dataframe: DataFrame zawierający dane (z kolumnami 'Material' i col_name)
    :param sap_column: Nazwa kolumny z numerami SAP w Excelu (np. 'A')
    :param frei_column: Nazwa kolumny w Excelu, do której mają być wpisywane dane (np. 'K')
    :param header_start: Nagłówek wskazujący początek zakresu (np. 'Stock')
    :param header_end: Nagłówek wskazujący koniec zakresu (np. 'S.C')
    """
    # Załadowanie istniejącego pliku Excel
    workbook = load_workbook(filename=file_path, keep_vba=True)
    # sheet = workbook.active  # Zakładamy, że pracujemy na aktywnym arkuszu
    sheet = workbook[sheet_name]

    # Znalezienie zakresów wierszy na podstawie nagłówków
    start_row = None
    end_row = None

    for row in sheet.iter_rows():
        cell_a_value = row[0].value  # Wartość w kolumnie A (zakładamy, że A to sap_column)
        if cell_a_value == header_start:
            start_row = row[0].row + 1  # Zakres zaczyna się od wiersza poniżej nagłówka
        elif cell_a_value == header_end:
            end_row = row[0].row - 1  # Kończy się przed wierszem z nagłówkiem
            break

    if start_row is None or end_row is None:
        print("Nie znaleziono odpowiednich nagłówków w pliku Excel.")
        return

    # Przekształcenie DataFrame na dictionary dla szybkiego wyszukiwania
    data_mapping = dict(zip(dataframe['Material'], dataframe[col_name]))

    # Iteracja po podanym zakresie wierszy w Excelu
    for row in range(start_row, end_row + 1):
        sap_value = str(sheet[f'{sap_column}{row}'].value)  # Pobierz wartość z kolumny SAP
        if sap_value in data_mapping:  # Jeśli wartość SAP występuje w DataFrame
            sheet[f'{frei_column}{row}'] = data_mapping[sap_value]  # Wpisz wartość z kolumny col_name
        else:
            sheet[f'{frei_column}{row}'] = None

    # Zapisanie pliku Excel
    workbook.save(file_path)
    print(f"Plik {file_path} został zaktualizowany.")

def get_z_mat_consumption_data(z_mat_file_path, dtypes, movements):
    df = pd.read_excel(z_mat_file_path, dtype=dtypes)
    df = df[df['movement'].isin(movements)]
    df = df.groupby(['Material', 'date'], as_index=False)['quantity'].sum()
    df['date'] = pd.to_datetime(df['date']).dt.date  # Konwersja na `datetime.date`

    return df

def get_mb51_consumption_data(z_mat_file_path, dtypes, col_names):
    df = pd.read_excel(z_mat_file_path, dtype=dtypes)
    df = df.rename(columns=col_names)
    df = df.groupby(['Material', 'date'], as_index=False)['quantity'].sum()
    df['date'] = pd.to_datetime(df['date']).dt.date  # Konwersja na `datetime.date`

    return df

def get_past_workdays(start_date, num_days, mode="last", country_holidays=None):
    """
    mode: "last" --> days before the start date, "next" --> days after the start date
    Zwraca listę dat num_days dni roboczych wstecz od start_date.
    Pomija weekendy i opcjonalnie święta.
    """
    workdays = []
    current_date = start_date

    while len(workdays) < num_days:
        if mode == 'last':
            current_date -= datetime.timedelta(days=1)
        elif mode == "next":
            current_date += datetime.timedelta(days=1)
        # Sprawdź czy to dzień roboczy
        if current_date.weekday() < 5:  # poniedziałek=0, piątek=4
            if country_holidays is None or current_date not in country_holidays:
                workdays.append(current_date)

    return workdays

def filter_z_mat_consumption_data(consumption_df, mat_list, days_range):
    consumption_df_filtered = consumption_df[(consumption_df['date'].isin(days_range)) & (consumption_df['Material'].isin(mat_list))]
    consumption_df_grouped = consumption_df_filtered.groupby('Material', as_index=False)['quantity'].sum()

    return consumption_df_grouped

def extract_unique_mb51_files(data_dict: dict) -> dict:
    result = {}

    for file_list in data_dict.values():
        if not file_list:
            continue

        key = file_list[0]

        if key not in result:
            result[key] = []

        for item in file_list:
            if item not in result[key]:
                result[key].append(item)

    return result

def load_grouped_mb51_dataframes(
    grouped_files: dict,
    base_path: str,
    extension: str = ".xlsx",
    dtypes_mb51: dict | None = None,
    col_names_mb51: dict | None = None,
    ignore_missing: bool = False,
) -> dict:
    """
    Load and combine files into DataFrames grouped by key.

    :param dtypes:
    :param grouped_files: dict, e.g. {"CONS_246": ["CONS_246", "CONS_246_2"]}
    :param base_path: directory path containing the files
    :param extension: file extension (".xlsx", ".csv", etc.)
    :param ignore_missing: if True, skip missing files instead of raising error
    :return: dict[str, pd.DataFrame]
    """

    base_path = Path(base_path)
    result = {}

    for group_key, file_list in grouped_files.items():
        print(f"Processing {group_key}")
        dataframes = []

        for file_name in file_list:
            file_path = base_path / f"{file_name}{extension}"

            if not file_path.exists():
                if ignore_missing:
                    continue
                else:
                    raise FileNotFoundError(f"File not found: {file_path}")

            # Select appropriate loader
            if extension.upper() == ".XLSX":
                df = get_mb51_consumption_data(file_path, dtypes_mb51, col_names_mb51)
            else:
                raise ValueError(f"Unsupported file extension: {extension}")

            # Add metadata columns (useful for debugging / traceability)
            df["source_file"] = file_name
            df["group_key"] = group_key

            dataframes.append(df)

        # Combine all DataFrames for the group
        if dataframes:
            result[group_key] = pd.concat(dataframes, ignore_index=True)
        else:
            result[group_key] = pd.DataFrame()

    return result

def retrieve_supplier_name(file_path: str) -> str:
    """
    Extract supplier name from full file path.

    Example:
    'P:\\...\\PurchAutomation_2101_ABC_COLORE.xlsm' -> 'ABC_COLORE'
    """

    file_name = Path(file_path).name  # extract file name

    start = file_name.find("_")
    end = file_name.rfind(".")

    if start == -1 or end == -1 or start >= end:
        raise ValueError(f"Invalid file name format: {file_path}")

    return file_name[start + 6:end]

### Script

In [6]:
# Updating the open orders data

movement_types = ('261', '313')

zek103_content = pd.read_excel(zek103_file_path, dtype=zek103_dtypes)
mb52df = get_mb52_data(mb52_file_path, mb52_dtypes)

# TODO: Get dfs for all exported files
unique_mb51_files_dict = extract_unique_mb51_files(supplier_files_dict)
MB551_dfs = load_grouped_mb51_dataframes(
    grouped_files=unique_mb51_files_dict,
    base_path=export_files_directory_path,
    extension=".xlsx",
    dtypes_mb51=mb51_consumption_dtypes,
    col_names_mb51=MB51_new_col_names
)


# TODO: Change path
supplier_files_paths = list_excel_files(supplier_files_directory_path_test)
excel_sheet = "2101_data"
plant = '2101',

storage_locs = ('0003', '0004', '0005', '0007')

current_year = datetime.date.today().year
years_list = [current_year - 1, current_year, current_year + 1]
pl_holidays = holidays.Poland(years=years_list)

today = datetime.datetime.today().date()
year_ago = today - datetime.timedelta(days=365)

past_20_days_now = get_past_workdays(today, 20, 'last', pl_holidays)
past_20_days_year_ago = get_past_workdays(year_ago, 20, 'last', pl_holidays)
next_20_days_year_ago = get_past_workdays(year_ago, 20, 'next', pl_holidays)
past_60_days_now = get_past_workdays(today, 60, 'last', pl_holidays)
past_60_days_year_ago = get_past_workdays(year_ago, 60, 'last', pl_holidays)

usage_parameters = [(next_20_days_year_ago, 'C'),
                    (past_20_days_now, 'D'),
                    (past_20_days_year_ago, 'E'),
                    (past_60_days_now, 'F'),
                    (past_60_days_year_ago, 'G'),
]

for file_path in supplier_files_paths:

    # TODO: Retrive supplier name from file path and get corresponding df
    supplier_name = retrieve_supplier_name(file_path)
    z_mat_or_mb51_consumption_df = MB551_dfs[supplier_files_dict[supplier_name][0]]

    sap_list = get_supplier_sap_numbers(file_path, excel_sheet)
    zek103_output = get_zek103_data(zek103_content, plant, sap_list)
    update_excel_with_quantities(file_path, zek103_output, 'SAP', 'CONSUMPTION', excel_sheet, True)

    mb52 = filter_mb52_data(mb52df, sap_list, '2101', storage_locs)
    # Wywołanie funkcji
    update_excel_with_dataframe(
        file_path=file_path,
        dataframe=mb52,
        sap_column='A',  # Kolumna z numerami SAP w Excelu
        frei_column='K',  # Kolumna, do której wpisywane są dane
        header_start='Stock',  # Nagłówek wskazujący początek
        header_end='S.C',       # Nagłówek wskazujący koniec
        sheet_name=excel_sheet,
    )

    for parameter in usage_parameters:
        z_mat_consumption_grouped = filter_z_mat_consumption_data(z_mat_or_mb51_consumption_df, sap_list, parameter[0])

        # wpisanie danych do Excela
        update_excel_with_dataframe(
            file_path=file_path,
            dataframe=z_mat_consumption_grouped,
            sap_column='A',  # Kolumna z numerami SAP w Excelu
            frei_column=parameter[1],  # Kolumna, do której wpisywane są dane
            header_start='Consumption',  # Nagłówek wskazujący początek
            header_end='Stock',       # Nagłówek wskazujący koniec
            sheet_name=excel_sheet,
            col_name='quantity'
        )

Excel file PurchAutomation_BACCARAT.xlsm updated successfully.
Plik P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_BACCARAT.xlsm został zaktualizowany.
Plik P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_BACCARAT.xlsm został zaktualizowany.
Plik P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_BACCARAT.xlsm został zaktualizowany.
Plik P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_BACCARAT.xlsm został zaktualizowany.
Plik P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_BACCARAT.xlsm został zaktualizowany.
Plik P:\Technisch\PLANY PRODUKCJI\PLANIŚCI\PP_TOOLS_TEMP_FILES\12_PURCHASE_AUTOMATION\supplier_files\PurchAutomation_BACCARAT.xlsm został zaktualizowany.
Excel file Pu

In [5]:
supplier_files_paths = list_excel_files(supplier_files_directory_path)
supplier_files_paths

['P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_ABC_COLORE.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_BACCARAT.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_BELATRONIC.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_BIZEA.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_BRETTHAUER.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_DAFA.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_DEVENTER.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_DOMICET.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_EJOT.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_FRANZEN.xlsm',
 'P:\\Zakupy\\O\\SupplierAutomation\\supplier_files\\PurchAutomation_2101_GALLARDO.xlsm',
 'P:\\Zakupy\\O\\

In [10]:
retrieve_supplier_name(supplier_files_paths[0])

'ABC_COLORE'